In [ ]:
"""
Gene Expression Batch Correction Analysis - Enhanced with Specific Visualizations
Version 2.2 - Added Silhouette Score and Distance Ratio plots
"""

import tkinter as tk
from tkinter import ttk, filedialog, messagebox, scrolledtext
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg, NavigationToolbar2Tk
from matplotlib.figure import Figure
from matplotlib.gridspec import GridSpec
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, silhouette_samples, calinski_harabasz_score, davies_bouldin_score
from sklearn.manifold import TSNE
from sklearn.neighbors import NearestNeighbors
import os
import sys
import threading
import warnings
import json
import pickle
from datetime import datetime
from pathlib import Path
from scipy.spatial.distance import cdist, pdist, squareform
from scipy import stats
warnings.filterwarnings('ignore')

# Try to import combat, install if needed
try:
    from combat.pycombat import pycombat
except ImportError:
    import subprocess
    print("Installing combat package...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "combat"])
    from combat.pycombat import pycombat

# ==================== PUBLICATION-QUALITY STYLING ====================
def set_publication_style():
    """Set publication-quality matplotlib parameters"""
    plt.style.use('default')
    
    # Core styling
    mpl.rcParams['figure.dpi'] = 100
    mpl.rcParams['savefig.dpi'] = 600
    mpl.rcParams['savefig.format'] = 'pdf'
    mpl.rcParams['savefig.bbox'] = 'tight'
    mpl.rcParams['savefig.pad_inches'] = 0.1
    
    # Font settings
    mpl.rcParams['font.family'] = 'sans-serif'
    mpl.rcParams['font.sans-serif'] = ['Arial', 'Helvetica', 'DejaVu Sans']
    mpl.rcParams['font.size'] = 10
    mpl.rcParams['axes.titlesize'] = 12
    mpl.rcParams['axes.labelsize'] = 11
    mpl.rcParams['xtick.labelsize'] = 9
    mpl.rcParams['ytick.labelsize'] = 9
    mpl.rcParams['legend.fontsize'] = 9
    
    # Line and marker settings
    mpl.rcParams['lines.linewidth'] = 1.5
    mpl.rcParams['lines.markersize'] = 6
    mpl.rcParams['axes.linewidth'] = 1.2
    mpl.rcParams['grid.linewidth'] = 0.8
    mpl.rcParams['grid.alpha'] = 0.3
    mpl.rcParams['grid.linestyle'] = '--'
    
    # Tick settings
    mpl.rcParams['xtick.major.width'] = 1.2
    mpl.rcParams['ytick.major.width'] = 1.2
    
    # Figure size presets
    FIGURE_SIZES = {
        'standard': (8, 6),
        'wide': (12, 5),
        'tall': (6, 8),
        'large': (14, 10),
        'square': (8, 8),
        'double': (16, 6),
        'metrics': (10, 8)
    }
    
    return FIGURE_SIZES

# Initialize style
FIGURE_SIZES = set_publication_style()

# Color palettes
COLOR_PALETTES = {
    'categorical': ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd'],
    'batches': ['#E41A1C', '#377EB8', '#4DAF4A', '#984EA3', '#FF7F00']
}

class EnhancedBatchCorrectionGUI:
    def __init__(self, root):
        self.root = root
        self.root.title("Gene Expression Batch Correction Analysis")
        self.root.geometry("1400x900")
        
        # Data storage
        self.dataframes = []
        self.file_names = []
        self.batch_labels = []
        self.sample_names = []
        self.df_expression = None
        self.df_corrected = None
        self.metrics = {}
        self.output_dir = None
        self.project_dir = None
        
        # Visualization settings
        self.current_figures = {}
        
        # Setup GUI
        self.setup_gui()
        
    def setup_gui(self):
        """Create enhanced GUI layout"""
        # Configure main window grid
        self.root.columnconfigure(0, weight=1)
        self.root.rowconfigure(0, weight=1)
        
        # Main container
        main_frame = ttk.Frame(self.root, padding="10")
        main_frame.grid(row=0, column=0, sticky=(tk.W, tk.E, tk.N, tk.S))
        main_frame.columnconfigure(0, weight=1)
        main_frame.rowconfigure(1, weight=1)
        
        # Header
        header_frame = ttk.Frame(main_frame)
        header_frame.grid(row=0, column=0, sticky=(tk.W, tk.E), pady=(0, 10))
        
        title_label = ttk.Label(header_frame, 
                               text="📊 Gene Expression Batch Correction Analysis",
                               font=('Arial', 18, 'bold'))
        title_label.pack(side=tk.LEFT)
        
        # Main content area
        content_frame = ttk.Frame(main_frame)
        content_frame.grid(row=1, column=0, sticky=(tk.W, tk.E, tk.N, tk.S))
        content_frame.columnconfigure(0, weight=1)
        content_frame.rowconfigure(0, weight=1)
        
        # Left panel - Controls
        control_panel = ttk.LabelFrame(content_frame, text="Control Panel", padding="15")
        control_panel.grid(row=0, column=0, sticky=(tk.W, tk.E, tk.N, tk.S), padx=(0, 10))
        control_panel.columnconfigure(0, weight=1)
        
        # File operations
        file_frame = ttk.LabelFrame(control_panel, text="Data Input", padding="10")
        file_frame.grid(row=0, column=0, sticky=(tk.W, tk.E), pady=(0, 10))
        
        ttk.Button(file_frame, text="📁 Load Expression Data (CSV)", 
                  command=self.load_expression_data,
                  width=25).pack(fill=tk.X, pady=2)
        
        self.file_info_label = ttk.Label(file_frame, text="No data loaded", 
                                        foreground="blue", wraplength=250)
        self.file_info_label.pack(pady=5)
        
        # Preprocessing options
        prep_frame = ttk.LabelFrame(control_panel, text="Preprocessing", padding="10")
        prep_frame.grid(row=1, column=0, sticky=(tk.W, tk.E), pady=(0, 10))
        
        # Log transformation
        log_frame = ttk.Frame(prep_frame)
        log_frame.pack(fill=tk.X, pady=2)
        
        self.apply_log = tk.BooleanVar(value=True)
        ttk.Checkbutton(log_frame, text="Apply Log2 Transformation", 
                       variable=self.apply_log).pack(side=tk.LEFT)
        
        ttk.Label(log_frame, text="Pseudo-count:").pack(side=tk.LEFT, padx=(10, 2))
        self.pseudo_count = ttk.Entry(log_frame, width=8)
        self.pseudo_count.insert(0, "1.0")
        self.pseudo_count.pack(side=tk.LEFT)
        
        # Batch correction
        batch_frame = ttk.LabelFrame(control_panel, text="Batch Correction", padding="10")
        batch_frame.grid(row=2, column=0, sticky=(tk.W, tk.E), pady=(0, 10))
        
        ttk.Button(batch_frame, text="🔄 Run Full Analysis", 
                  command=self.run_full_analysis,
                  style="Accent.TButton").pack(fill=tk.X, pady=2)
        
        # Visualization options - SPECIFIC VISUALIZATIONS YOU REQUESTED
        viz_frame = ttk.LabelFrame(control_panel, text="Key Visualizations", padding="10")
        viz_frame.grid(row=3, column=0, sticky=(tk.W, tk.E), pady=(0, 10))
        
        # Your requested visualizations
        viz_options = [
            ("📊 Silhouette Score Plot", self.generate_silhouette_plot),
            ("📈 Distance Ratio Plot", self.generate_distance_ratio_plot),
            ("🔬 PCA Analysis", self.generate_pca_plot),
            ("📉 All Metrics Plots", self.generate_all_metrics_plots)
        ]
        
        for text, command in viz_options:
            ttk.Button(viz_frame, text=text, command=command).pack(fill=tk.X, pady=2)
        
        # Export options
        export_frame = ttk.LabelFrame(control_panel, text="Export Results", padding="10")
        export_frame.grid(row=4, column=0, sticky=(tk.W, tk.E))
        
        ttk.Button(export_frame, text="📂 Set Output Directory", 
                  command=self.set_output_directory).pack(fill=tk.X, pady=2)
        
        ttk.Button(export_frame, text="💾 Save All Results", 
                  command=self.save_all_results).pack(fill=tk.X, pady=2)
        
        # Right panel - Main display
        display_panel = ttk.Frame(content_frame)
        display_panel.grid(row=0, column=1, sticky=(tk.W, tk.E, tk.N, tk.S))
        display_panel.columnconfigure(0, weight=1)
        display_panel.rowconfigure(0, weight=1)
        
        # Notebook for tabs
        self.notebook = ttk.Notebook(display_panel)
        self.notebook.grid(row=0, column=0, sticky=(tk.W, tk.E, tk.N, tk.S))
        
        # Log tab
        log_tab = ttk.Frame(self.notebook)
        self.notebook.add(log_tab, text="📋 Analysis Log")
        
        self.log_text = scrolledtext.ScrolledText(log_tab, height=20, width=80,
                                                 font=('Consolas', 9))
        self.log_text.pack(fill=tk.BOTH, expand=True, padx=5, pady=5)
        
        # Status bar
        status_frame = ttk.Frame(main_frame)
        status_frame.grid(row=2, column=0, columnspan=2, sticky=(tk.W, tk.E), pady=(5, 0))
        
        self.progress = ttk.Progressbar(status_frame, mode='indeterminate')
        self.progress.pack(side=tk.LEFT, fill=tk.X, expand=True, padx=(0, 10))
        
        self.status_label = ttk.Label(status_frame, text="Ready", 
                                     relief=tk.SUNKEN, anchor=tk.W, padding=5)
        self.status_label.pack(side=tk.LEFT, fill=tk.X, expand=True)
        
        # Apply custom styles
        self.setup_styles()
        
    def setup_styles(self):
        """Setup custom ttk styles"""
        style = ttk.Style()
        style.theme_use('clam')
        
        style.configure('Accent.TButton', 
                       foreground='white', 
                       background='#0078D7',
                       font=('Arial', 10, 'bold'))
        
    def log_message(self, message, level="INFO"):
        """Add timestamped message to log"""
        timestamp = datetime.now().strftime("%H:%M:%S")
        self.log_text.insert(tk.END, f"[{timestamp}] {message}\n")
        self.log_text.see(tk.END)
        self.root.update_idletasks()
        
    def update_status(self, message):
        """Update status bar"""
        self.status_label.config(text=message)
        self.root.update_idletasks()
        
    def load_expression_data(self):
        """Load expression data from CSV files"""
        file_paths = filedialog.askopenfilenames(
            title="Select Expression Data Files",
            filetypes=[("CSV files", "*.csv"), ("All files", "*.*")]
        )
        
        if not file_paths:
            return
            
        self.log_message(f"Loading {len(file_paths)} expression data files...")
        
        self.dataframes = []
        self.batch_labels = []
        self.sample_names = []
        batch_counter = 0
        
        for file_path in file_paths:
            try:
                df = pd.read_csv(file_path, index_col=0)
                df.columns = df.columns.str.strip()
                
                if df.index.duplicated().any():
                    df = df.groupby(df.index).mean()
                
                self.dataframes.append(df)
                self.file_names.append(Path(file_path).name)
                self.batch_labels.extend([batch_counter] * len(df.columns))
                self.sample_names.extend(df.columns.tolist())
                
                self.log_message(f"  ✓ {Path(file_path).name}: {df.shape[0]} genes × {df.shape[1]} samples")
                batch_counter += 1
                
            except Exception as e:
                self.log_message(f"  ✗ Error loading {Path(file_path).name}: {str(e)}")
                messagebox.showerror("Load Error", f"Failed to load {Path(file_path).name}:\n{str(e)}")
                return
        
        # Merge all dataframes
        self.log_message("Merging datasets...")
        self.df_expression = pd.concat(self.dataframes, axis=1, join='inner')
        
        self.log_message(f"Merged dataset: {self.df_expression.shape[0]} genes × {self.df_expression.shape[1]} samples")
        
        info_text = f"Loaded: {len(file_paths)} files\n"
        info_text += f"Genes: {self.df_expression.shape[0]:,}\n"
        info_text += f"Samples: {self.df_expression.shape[1]:,}\n"
        info_text += f"Batches: {len(set(self.batch_labels))}"
        self.file_info_label.config(text=info_text)
        
        messagebox.showinfo("Success", 
                          f"Successfully loaded {len(file_paths)} files\n"
                          f"Common genes: {self.df_expression.shape[0]:,}\n"
                          f"Total samples: {self.df_expression.shape[1]:,}")
        
    def preprocess_data(self, data):
        """Apply preprocessing steps"""
        processed = data.copy()
        
        # Log transformation
        if self.apply_log.get():
            try:
                pseudo_count = float(self.pseudo_count.get())
                processed = np.log2(processed + pseudo_count)
                self.log_message(f"Applied log2 transformation (pseudo-count: {pseudo_count})")
            except ValueError:
                self.log_message("Invalid pseudo-count value")
                return None
        
        return processed
    
    def calculate_within_between_distance_ratio(self, data):
        """Calculate within-batch vs between-batch distance ratio"""
        unique_batches = np.unique(self.batch_labels)
        
        # Calculate pairwise distances
        scaled_data = StandardScaler().fit_transform(data.T)
        distances = squareform(pdist(scaled_data, metric='euclidean'))
        
        within_distances = []
        between_distances = []
        
        for i in range(len(self.batch_labels)):
            for j in range(i+1, len(self.batch_labels)):
                if self.batch_labels[i] == self.batch_labels[j]:
                    within_distances.append(distances[i, j])
                else:
                    between_distances.append(distances[i, j])
        
        if len(within_distances) > 0 and len(between_distances) > 0:
            mean_within = np.mean(within_distances)
            mean_between = np.mean(between_distances)
            ratio = mean_within / mean_between
        else:
            ratio = np.nan
        
        return ratio, mean_within, mean_between
    
    def calculate_silhouette_scores(self, data):
        """Calculate silhouette scores for each sample"""
        scaled_data = StandardScaler().fit_transform(data.T)
        
        # Overall silhouette score
        overall_score = silhouette_score(scaled_data, self.batch_labels)
        
        # Individual silhouette scores
        sample_scores = silhouette_samples(scaled_data, self.batch_labels)
        
        return overall_score, sample_scores
    
    def calculate_metrics(self, raw_data, corrected_data):
        """Calculate comprehensive batch correction metrics"""
        self.log_message("Calculating batch correction metrics...")
        
        metrics = {}
        
        # Calculate silhouette scores
        try:
            metrics['raw_silhouette'], raw_sample_scores = self.calculate_silhouette_scores(raw_data)
            metrics['corrected_silhouette'], corr_sample_scores = self.calculate_silhouette_scores(corrected_data)
            metrics['silhouette_improvement'] = metrics['corrected_silhouette'] - metrics['raw_silhouette']
            
            # Store sample scores for visualization
            self.raw_sample_scores = raw_sample_scores
            self.corr_sample_scores = corr_sample_scores
        except:
            metrics['raw_silhouette'] = np.nan
            metrics['corrected_silhouette'] = np.nan
            metrics['silhouette_improvement'] = np.nan
        
        # Calculate distance ratios
        metrics['raw_distance_ratio'], metrics['raw_mean_within'], metrics['raw_mean_between'] = \
            self.calculate_within_between_distance_ratio(raw_data)
        
        metrics['corrected_distance_ratio'], metrics['corrected_mean_within'], metrics['corrected_mean_between'] = \
            self.calculate_within_between_distance_ratio(corrected_data)
        
        # Store for visualization
        self.distance_metrics = {
            'raw': (metrics['raw_distance_ratio'], metrics['raw_mean_within'], metrics['raw_mean_between']),
            'corrected': (metrics['corrected_distance_ratio'], metrics['corrected_mean_within'], metrics['corrected_mean_between'])
        }
        
        self.log_message("Metrics calculation complete")
        return metrics
    
    def run_full_analysis(self):
        """Run the complete analysis pipeline"""
        if self.df_expression is None:
            messagebox.showwarning("Warning", "Please load expression data first")
            return
            
        self.progress.start()
        self.update_status("Running full analysis pipeline...")
        
        thread = threading.Thread(target=self._analysis_worker)
        thread.start()
        
    def _analysis_worker(self):
        """Worker thread for analysis"""
        try:
            # Step 1: Preprocessing
            self.log_message("\n=== STEP 1: Preprocessing ===")
            processed_data = self.preprocess_data(self.df_expression)
            if processed_data is None:
                return
            
            # Step 2: Batch Correction
            self.log_message("\n=== STEP 2: Batch Correction ===")
            self.log_message("Running ComBat batch correction...")
            
            batch_labels_numeric = [int(x) for x in self.batch_labels]
            self.df_corrected = pycombat(processed_data, batch_labels_numeric)
            self.log_message("Batch correction complete")
            
            # Step 3: Calculate Metrics
            self.log_message("\n=== STEP 3: Metrics Calculation ===")
            self.metrics = self.calculate_metrics(processed_data, self.df_corrected)
            
            # Display metrics
            self.display_metrics_summary()
            
            self.root.after(0, self.update_status, "Analysis complete!")
            self.root.after(0, lambda: messagebox.showinfo("Success", 
                "Full analysis completed successfully!"))
            
        except Exception as e:
            self.log_message(f"\nError during analysis: {str(e)}")
            self.root.after(0, lambda: messagebox.showerror("Error", 
                f"Analysis failed:\n{str(e)}"))
        finally:
            self.root.after(0, self.progress.stop)
    
    def display_metrics_summary(self):
        """Display metrics summary in log"""
        self.log_message("\n=== BATCH CORRECTION METRICS ===")
        
        if 'raw_silhouette' in self.metrics:
            self.log_message(f"Silhouette Score:")
            self.log_message(f"  Before: {self.metrics['raw_silhouette']:.3f}")
            self.log_message(f"  After:  {self.metrics['corrected_silhouette']:.3f}")
            self.log_message(f"  Improvement: {self.metrics['silhouette_improvement']:+.3f}")
        
        if 'raw_distance_ratio' in self.metrics:
            self.log_message(f"\nWithin/Between Batch Distance Ratio:")
            self.log_message(f"  Before: {self.metrics['raw_distance_ratio']:.3f}")
            self.log_message(f"  After:  {self.metrics['corrected_distance_ratio']:.3f}")
            
            if self.metrics['raw_mean_within'] and self.metrics['raw_mean_between']:
                self.log_message(f"  Mean within-batch distance (before): {self.metrics['raw_mean_within']:.3f}")
                self.log_message(f"  Mean between-batch distance (before): {self.metrics['raw_mean_between']:.3f}")
                self.log_message(f"  Mean within-batch distance (after): {self.metrics['corrected_mean_within']:.3f}")
                self.log_message(f"  Mean between-batch distance (after): {self.metrics['corrected_mean_between']:.3f}")
    
    def generate_silhouette_plot(self):
        """Generate Silhouette Score visualization"""
        if self.df_corrected is None:
            messagebox.showwarning("Warning", "Please run analysis first")
            return
            
        self.log_message("Generating Silhouette Score plot...")
        
        fig = plt.figure(figsize=FIGURE_SIZES['wide'], dpi=100)
        
        # Data for bar plot
        conditions = ['Before Correction', 'After Correction']
        silhouette_scores = [
            self.metrics.get('raw_silhouette', 0),
            self.metrics.get('corrected_silhouette', 0)
        ]
        improvement = self.metrics.get('silhouette_improvement', 0)
        
        # Create bar plot
        ax1 = fig.add_subplot(121)
        bars = ax1.bar(conditions, silhouette_scores, 
                      color=['#E41A1C', '#377EB8'], alpha=0.8)
        
        # Add value labels on top of bars
        for i, (bar, score) in enumerate(zip(bars, silhouette_scores)):
            height = bar.get_height()
            ax1.text(bar.get_x() + bar.get_width()/2., height + 0.01,
                    f'{score:.3f}', ha='center', va='bottom', fontweight='bold')
        
        ax1.set_ylabel('Silhouette Score', fontweight='bold')
        ax1.set_title('Batch Clustering (Silhouette Score)', fontweight='bold', pad=15)
        ax1.set_ylim([-0.5, 1.0])  # Silhouette score range is -1 to 1
        ax1.grid(True, alpha=0.3, axis='y')
        
        # Add improvement arrow
        ax1.annotate(f'Δ = {improvement:+.3f}', 
                    xy=(1, silhouette_scores[1]), 
                    xytext=(0.5, max(silhouette_scores) + 0.1),
                    arrowprops=dict(arrowstyle='->', color='green', lw=2),
                    ha='center', fontweight='bold', color='green')
        
        # Individual sample silhouette scores (violin plot)
        ax2 = fig.add_subplot(122)
        
        # Prepare data for violin plot
        silhouette_data = []
        if hasattr(self, 'raw_sample_scores'):
            silhouette_data.append(self.raw_sample_scores)
        if hasattr(self, 'corr_sample_scores'):
            silhouette_data.append(self.corr_sample_scores)
        
        if silhouette_data:
            vp = ax2.violinplot(silhouette_data, showmeans=True, showmedians=True)
            
            # Color the violins
            colors = ['#E41A1C', '#377EB8']
            for i, (body, color) in enumerate(zip(vp['bodies'], colors)):
                body.set_facecolor(color)
                body.set_alpha(0.7)
                body.set_edgecolor('black')
            
            # Color the lines
            for partname in ('cbars', 'cmins', 'cmaxes', 'cmeans', 'cmedians'):
                if partname in vp:
                    vp[partname].set_color('black')
                    vp[partname].set_linewidth(1.5)
        
        ax2.set_xticks([1, 2])
        ax2.set_xticklabels(['Before', 'After'])
        ax2.set_ylabel('Silhouette Score (per sample)', fontweight='bold')
        ax2.set_title('Distribution of Sample Silhouette Scores', fontweight='bold', pad=15)
        ax2.grid(True, alpha=0.3, axis='y')
        
        fig.suptitle('Silhouette Score Analysis', fontsize=14, fontweight='bold', y=0.98)
        fig.tight_layout(rect=[0, 0, 1, 0.95])
        
        # Display in GUI
        self.display_figure(fig, "Silhouette Score Analysis")
        self.log_message("Silhouette Score plot generated successfully")
        
        return fig
    
    def generate_distance_ratio_plot(self):
        """Generate Within/Between Batch Distance Ratio visualization"""
        if self.df_corrected is None:
            messagebox.showwarning("Warning", "Please run analysis first")
            return
            
        self.log_message("Generating Distance Ratio plot...")
        
        fig = plt.figure(figsize=FIGURE_SIZES['wide'], dpi=100)
        
        # Data for bar plot
        conditions = ['Before Correction', 'After Correction']
        
        if hasattr(self, 'distance_metrics'):
            distance_ratios = [
                self.distance_metrics['raw'][0],
                self.distance_metrics['corrected'][0]
            ]
            mean_within = [
                self.distance_metrics['raw'][1],
                self.distance_metrics['corrected'][1]
            ]
            mean_between = [
                self.distance_metrics['raw'][2],
                self.distance_metrics['corrected'][2]
            ]
        else:
            # Fallback to metrics
            distance_ratios = [
                self.metrics.get('raw_distance_ratio', 1),
                self.metrics.get('corrected_distance_ratio', 1)
            ]
            mean_within = [1, 1]
            mean_between = [2, 2]
        
        # Create bar plot for distance ratio
        ax1 = fig.add_subplot(121)
        bars = ax1.bar(conditions, distance_ratios, 
                      color=['#E41A1C', '#377EB8'], alpha=0.8)
        
        # Add value labels on top of bars
        for i, (bar, ratio) in enumerate(zip(bars, distance_ratios)):
            height = bar.get_height()
            ax1.text(bar.get_x() + bar.get_width()/2., height + 0.05,
                    f'{ratio:.3f}', ha='center', va='bottom', fontweight='bold')
        
        ax1.set_ylabel('Within/Between Batch Distance Ratio', fontweight='bold')
        ax1.set_title('Distance Ratio Analysis', fontweight='bold', pad=15)
        ax1.axhline(y=1.0, color='red', linestyle='--', alpha=0.5, label='Ideal (Ratio = 1)')
        ax1.legend()
        ax1.grid(True, alpha=0.3, axis='y')
        
        # Bar plot for mean distances
        ax2 = fig.add_subplot(122)
        
        x = np.arange(len(conditions))
        width = 0.35
        
        bars1 = ax2.bar(x - width/2, mean_within, width, 
                       label='Mean Within-Batch Distance', 
                       color='#4DAF4A', alpha=0.8)
        bars2 = ax2.bar(x + width/2, mean_between, width, 
                       label='Mean Between-Batch Distance', 
                       color='#984EA3', alpha=0.8)
        
        # Add value labels
        for bars in [bars1, bars2]:
            for bar in bars:
                height = bar.get_height()
                ax2.text(bar.get_x() + bar.get_width()/2., height + 0.05,
                        f'{height:.2f}', ha='center', va='bottom', fontsize=8)
        
        ax2.set_xticks(x)
        ax2.set_xticklabels(conditions)
        ax2.set_ylabel('Mean Distance', fontweight='bold')
        ax2.set_title('Mean Within vs Between Batch Distances', fontweight='bold', pad=15)
        ax2.legend()
        ax2.grid(True, alpha=0.3, axis='y')
        
        fig.suptitle('Batch Distance Analysis', fontsize=14, fontweight='bold', y=0.98)
        fig.tight_layout(rect=[0, 0, 1, 0.95])
        
        # Display in GUI
        self.display_figure(fig, "Distance Ratio Analysis")
        self.log_message("Distance Ratio plot generated successfully")
        
        return fig
    
    def generate_pca_plot(self):
        """Generate PCA visualization"""
        if self.df_corrected is None:
            messagebox.showwarning("Warning", "Please run analysis first")
            return
            
        self.log_message("Generating PCA plot...")
        
        fig = plt.figure(figsize=FIGURE_SIZES['double'], dpi=100)
        
        unique_batches = np.unique(self.batch_labels)
        batch_colors = {batch: COLOR_PALETTES['batches'][i % len(COLOR_PALETTES['batches'])] 
                       for i, batch in enumerate(unique_batches)}
        
        # Raw data PCA
        ax1 = fig.add_subplot(121)
        pca_raw = PCA(n_components=2)
        raw_scaled = StandardScaler().fit_transform(self.df_expression.T)
        pca_result_raw = pca_raw.fit_transform(raw_scaled)
        var_raw = pca_raw.explained_variance_ratio_ * 100
        
        for batch in unique_batches:
            batch_idx = [i for i, x in enumerate(self.batch_labels) if x == batch]
            ax1.scatter(pca_result_raw[batch_idx, 0], pca_result_raw[batch_idx, 1],
                       c=batch_colors[batch], label=f'Batch {batch}', 
                       s=60, alpha=0.7, edgecolors='w', linewidth=0.5)
        
        ax1.set_title('Before Correction', fontweight='bold', pad=15)
        ax1.set_xlabel(f'PC1 ({var_raw[0]:.1f}%)', fontweight='bold')
        ax1.set_ylabel(f'PC2 ({var_raw[1]:.1f}%)', fontweight='bold')
        ax1.grid(True, alpha=0.3, linestyle='--')
        ax1.legend()
        
        # Corrected data PCA
        ax2 = fig.add_subplot(122)
        pca_corr = PCA(n_components=2)
        corr_scaled = StandardScaler().fit_transform(self.df_corrected.T)
        pca_result_corr = pca_corr.fit_transform(corr_scaled)
        var_corr = pca_corr.explained_variance_ratio_ * 100
        
        for batch in unique_batches:
            batch_idx = [i for i, x in enumerate(self.batch_labels) if x == batch]
            ax2.scatter(pca_result_corr[batch_idx, 0], pca_result_corr[batch_idx, 1],
                       c=batch_colors[batch], label=f'Batch {batch}', 
                       s=60, alpha=0.7, edgecolors='w', linewidth=0.5)
        
        ax2.set_title('After Correction', fontweight='bold', pad=15)
        ax2.set_xlabel(f'PC1 ({var_corr[0]:.1f}%)', fontweight='bold')
        ax2.set_ylabel(f'PC2 ({var_corr[1]:.1f}%)', fontweight='bold')
        ax2.grid(True, alpha=0.3, linestyle='--')
        ax2.legend()
        
        fig.suptitle('PCA Analysis of Batch Correction', fontsize=14, fontweight='bold')
        fig.tight_layout()
        
        self.display_figure(fig, "PCA Analysis")
        self.log_message("PCA plot generated successfully")
        
        return fig
    
    def generate_all_metrics_plots(self):
        """Generate all metrics plots in one figure"""
        if self.df_corrected is None:
            messagebox.showwarning("Warning", "Please run analysis first")
            return
            
        self.log_message("Generating all metrics plots...")
        
        fig = plt.figure(figsize=FIGURE_SIZES['large'], dpi=100)
        
        # Create 2x2 grid for plots
        gs = GridSpec(2, 2, figure=fig, hspace=0.3, wspace=0.3)
        
        # 1. Silhouette Score bar plot (top-left)
        ax1 = fig.add_subplot(gs[0, 0])
        conditions = ['Before', 'After']
        silhouette_scores = [
            self.metrics.get('raw_silhouette', 0),
            self.metrics.get('corrected_silhouette', 0)
        ]
        
        bars1 = ax1.bar(conditions, silhouette_scores, 
                       color=['#E41A1C', '#377EB8'], alpha=0.8)
        
        for bar, score in zip(bars1, silhouette_scores):
            height = bar.get_height()
            ax1.text(bar.get_x() + bar.get_width()/2., height + 0.01,
                    f'{score:.3f}', ha='center', va='bottom', fontweight='bold')
        
        ax1.set_ylabel('Silhouette Score', fontweight='bold')
        ax1.set_title('Batch Clustering (Silhouette Score)', fontweight='bold')
        ax1.set_ylim([min(silhouette_scores) - 0.1, max(silhouette_scores) + 0.15])
        ax1.grid(True, alpha=0.3, axis='y')
        
        # 2. Distance Ratio bar plot (top-right)
        ax2 = fig.add_subplot(gs[0, 1])
        distance_ratios = [
            self.metrics.get('raw_distance_ratio', 1),
            self.metrics.get('corrected_distance_ratio', 1)
        ]
        
        bars2 = ax2.bar(conditions, distance_ratios, 
                       color=['#E41A1C', '#377EB8'], alpha=0.8)
        
        for bar, ratio in zip(bars2, distance_ratios):
            height = bar.get_height()
            ax2.text(bar.get_x() + bar.get_width()/2., height + 0.05,
                    f'{ratio:.3f}', ha='center', va='bottom', fontweight='bold')
        
        ax2.set_ylabel('Within/Between Distance Ratio', fontweight='bold')
        ax2.set_title('Distance Ratio Analysis', fontweight='bold')
        ax2.axhline(y=1.0, color='red', linestyle='--', alpha=0.5)
        ax2.grid(True, alpha=0.3, axis='y')
        
        # 3. Silhouette distribution (bottom-left)
        ax3 = fig.add_subplot(gs[1, 0])
        if hasattr(self, 'raw_sample_scores') and hasattr(self, 'corr_sample_scores'):
            silhouette_data = [self.raw_sample_scores, self.corr_sample_scores]
            vp = ax3.violinplot(silhouette_data, showmeans=True)
            
            colors = ['#E41A1C', '#377EB8']
            for i, (body, color) in enumerate(zip(vp['bodies'], colors)):
                body.set_facecolor(color)
                body.set_alpha(0.7)
        
        ax3.set_xticks([1, 2])
        ax3.set_xticklabels(conditions)
        ax3.set_ylabel('Silhouette Score', fontweight='bold')
        ax3.set_title('Sample Silhouette Distribution', fontweight='bold')
        ax3.grid(True, alpha=0.3, axis='y')
        
        # 4. Improvement metrics (bottom-right)
        ax4 = fig.add_subplot(gs[1, 1])
        
        improvements = {
            'Silhouette Score': self.metrics.get('silhouette_improvement', 0),
            'Distance Ratio': self.metrics.get('corrected_distance_ratio', 1) - self.metrics.get('raw_distance_ratio', 1)
        }
        
        colors = ['#2CA02C' if x > 0 else '#D62728' for x in improvements.values()]
        bars4 = ax4.bar(range(len(improvements)), list(improvements.values()), 
                       color=colors, alpha=0.8)
        
        ax4.set_xticks(range(len(improvements)))
        ax4.set_xticklabels(list(improvements.keys()), rotation=45, ha='right')
        ax4.set_ylabel('Improvement', fontweight='bold')
        ax4.set_title('Batch Correction Improvement', fontweight='bold')
        ax4.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
        ax4.grid(True, alpha=0.3, axis='y')
        
        # Add value labels
        for bar, value in zip(bars4, improvements.values()):
            height = bar.get_height()
            ax4.text(bar.get_x() + bar.get_width()/2., 
                    height + (0.01 if height >= 0 else -0.03),
                    f'{value:+.3f}', ha='center', va='bottom' if height >= 0 else 'top',
                    fontweight='bold')
        
        fig.suptitle('Batch Correction Metrics Summary', fontsize=16, fontweight='bold', y=0.98)
        fig.tight_layout(rect=[0, 0, 1, 0.95])
        
        self.display_figure(fig, "Metrics Summary")
        self.log_message("All metrics plots generated successfully")
        
        return fig
    
    def display_figure(self, fig, title):
        """Display matplotlib figure in a new tab"""
        if title in self.current_figures:
            self.notebook.forget(self.current_figures[title]['tab'])
            
        tab = ttk.Frame(self.notebook)
        self.notebook.add(tab, text=title)
        
        canvas = FigureCanvasTkAgg(fig, master=tab)
        canvas.draw()
        canvas.get_tk_widget().pack(fill=tk.BOTH, expand=True)
        
        toolbar = NavigationToolbar2Tk(canvas, tab)
        toolbar.update()
        toolbar.pack(side=tk.BOTTOM, fill=tk.X)
        
        self.current_figures[title] = {
            'fig': fig,
            'canvas': canvas,
            'tab': tab
        }
        
        self.notebook.select(tab)
    
    def set_output_directory(self):
        """Set output directory for saving results"""
        directory = filedialog.askdirectory(
            title="Select Output Directory"
        )
        
        if directory:
            self.output_dir = Path(directory)
            self.log_message(f"Output directory set to: {self.output_dir}")
            
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            self.project_dir = self.output_dir / f"BatchCorrection_{timestamp}"
            self.project_dir.mkdir(exist_ok=True)
            
            messagebox.showinfo("Output Directory", 
                              f"Results will be saved to:\n{self.project_dir}")
    
    def save_all_results(self):
        """Save all results to files"""
        if self.df_corrected is None:
            messagebox.showwarning("Warning", "No analysis results to save")
            return
            
        if not hasattr(self, 'project_dir'):
            self.set_output_directory()
            if not hasattr(self, 'project_dir'):
                return
            
        self.progress.start()
        self.update_status("Saving all results...")
        
        thread = threading.Thread(target=self._save_results_worker)
        thread.start()
    
    def _save_results_worker(self):
        """Worker thread to save all results"""
        try:
            # Create subdirectories
            data_dir = self.project_dir / "data"
            plots_dir = self.project_dir / "plots"
            reports_dir = self.project_dir / "reports"
            
            for dir_path in [data_dir, plots_dir, reports_dir]:
                dir_path.mkdir(exist_ok=True)
            
            self.log_message(f"\n=== Saving Results to {self.project_dir} ===")
            
            # 1. Save data files
            self.log_message("1. Saving data files...")
            
            orig_path = data_dir / "original_expression_data.csv"
            self.df_expression.to_csv(orig_path)
            self.log_message(f"  ✓ Original data: {orig_path}")
            
            corr_path = data_dir / "batch_corrected_data.csv"
            self.df_corrected.to_csv(corr_path)
            self.log_message(f"  ✓ Corrected data: {corr_path}")
            
            # 2. Save metrics
            self.log_message("2. Saving metrics...")
            
            metrics_json = reports_dir / "batch_correction_metrics.json"
            with open(metrics_json, 'w') as f:
                json.dump({k: (float(v) if isinstance(v, (np.floating, float)) else v) 
                          for k, v in self.metrics.items()}, f, indent=2)
            self.log_message(f"  ✓ Metrics (JSON): {metrics_json}")
            
            # 3. Save plots - YOUR REQUESTED VISUALIZATIONS
            self.log_message("3. Saving publication-quality plots...")
            
            plot_functions = [
                ("silhouette_score_analysis", self.generate_silhouette_plot),
                ("distance_ratio_analysis", self.generate_distance_ratio_plot),
                ("pca_analysis", self.generate_pca_plot),
                ("metrics_summary", self.generate_all_metrics_plots)
            ]
            
            saved_plots = []
            for plot_name, plot_func in plot_functions:
                try:
                    fig = plot_func()
                    if fig:
                        # Save as PNG (600 DPI for publication)
                        png_path = plots_dir / f"{plot_name}.png"
                        fig.savefig(png_path, dpi=600, bbox_inches='tight')
                        saved_plots.append(png_path.name)
                        
                        # Save as PDF (vector format for publication)
                        pdf_path = plots_dir / f"{plot_name}.pdf"
                        fig.savefig(pdf_path, format='pdf', bbox_inches='tight')
                        saved_plots.append(pdf_path.name)
                        
                        # Save as SVG (editable vector format)
                        svg_path = plots_dir / f"{plot_name}.svg"
                        fig.savefig(svg_path, format='svg', bbox_inches='tight')
                        saved_plots.append(svg_path.name)
                        
                        self.log_message(f"  ✓ {plot_name}: PNG/PDF/SVG")
                except Exception as e:
                    self.log_message(f"  ✗ Error saving {plot_name}: {str(e)}")
            
            # 4. Save analysis report
            self.log_message("4. Generating analysis report...")
            
            report_path = reports_dir / "analysis_report.txt"
            with open(report_path, 'w') as f:
                f.write("=" * 80 + "\n")
                f.write("BATCH CORRECTION ANALYSIS REPORT\n")
                f.write("=" * 80 + "\n\n")
                
                f.write(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
                f.write(f"Output directory: {self.project_dir}\n\n")
                
                f.write("KEY METRICS:\n")
                f.write("-" * 40 + "\n")
                
                if 'raw_silhouette' in self.metrics:
                    f.write(f"Silhouette Score:\n")
                    f.write(f"  Before correction: {self.metrics['raw_silhouette']:.3f}\n")
                    f.write(f"  After correction:  {self.metrics['corrected_silhouette']:.3f}\n")
                    f.write(f"  Improvement:       {self.metrics['silhouette_improvement']:+.3f}\n\n")
                
                if 'raw_distance_ratio' in self.metrics:
                    f.write(f"Within/Between Batch Distance Ratio:\n")
                    f.write(f"  Before correction: {self.metrics['raw_distance_ratio']:.3f}\n")
                    f.write(f"  After correction:  {self.metrics['corrected_distance_ratio']:.3f}\n\n")
                
                f.write("VISUALIZATIONS SAVED:\n")
                f.write("-" * 40 + "\n")
                for plot in saved_plots:
                    f.write(f"  • {plot}\n")
            
            self.log_message(f"  ✓ Analysis report: {report_path}")
            
            # Show summary
            summary = f"""Results saved successfully!

Location: {self.project_dir}

Contents:
• Data files (CSV format)
• Publication-quality visualizations (PNG/PDF/SVG at 600 DPI)
• Analysis reports

Your requested visualizations:
1. Silhouette Score Analysis
2. Distance Ratio Analysis
3. PCA Analysis
4. Metrics Summary

All plots are saved in multiple formats suitable for publication.
"""
            
            self.root.after(0, lambda: messagebox.showinfo("Save Complete", summary))
            self.root.after(0, self.update_status, "Results saved successfully")
            
            # Ask to open directory
            self.root.after(0, self.ask_open_directory)
            
        except Exception as e:
            self.log_message(f"\nError saving results: {str(e)}")
            self.root.after(0, lambda: messagebox.showerror("Save Error", 
                f"Failed to save results:\n{str(e)}"))
        finally:
            self.root.after(0, self.progress.stop)
    
    def ask_open_directory(self):
        """Ask user if they want to open the output directory"""
        response = messagebox.askyesno(
            "Open Directory",
            f"Results saved to:\n{self.project_dir}\n\nOpen this directory?"
        )
        
        if response:
            import subprocess
            import platform
            
            path = str(self.project_dir)
            
            if platform.system() == "Windows":
                os.startfile(path)
            elif platform.system() == "Darwin":
                subprocess.run(["open", path])
            else:
                subprocess.run(["xdg-open", path])


def main():
    """Main function to run the application"""
    root = tk.Tk()
    app = EnhancedBatchCorrectionGUI(root)
    
    # Center window
    root.update_idletasks()
    width = root.winfo_width()
    height = root.winfo_height()
    x = (root.winfo_screenwidth() // 2) - (width // 2)
    y = (root.winfo_screenheight() // 2) - (height // 2)
    root.geometry(f'{width}x{height}+{x}+{y}')
    
    root.mainloop()


if __name__ == "__main__":
    main()